# Data
Grab data from Yahoo Financce API using `yfinance` library.

In [4]:
!pip install -r ../requirements.txt

630.42s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 18.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 19.7 MB/s  0:00:00
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl size=15636 sha256=75405f7428ec4af25941d4bb572e9b27d81a603eacbc9deccdb9d2ea0a1f7898
  Stored in directory: /Users/palomaperezdemadrid/Library/Caches/pip/wheels/1e/df/0f/e2bbb22d689b30c681feb5410ab64a2523437b34c8ecfc6476
  Created wheel for peewee: filename=peewee-3.18.3-cp313-cp313-macosx_15_0_arm64.whl size=272298 sha256=c8b2b801e14cc5990b64c308a658e74a24c9a468791c49b9701c2f178c68e0d7
  Stored in directory: /Users/palomaperezdemadrid/Library/Caches/pip/wheels/8c/a9/a4/df972cd49f865ffde174d9c5b26

In [6]:
import yfinance as yf

def get_spot_price(ticker: str) -> float:
    """
    Fetch the latest available spot price from Yahoo Finance.
    """
    stock = yf.Ticker(ticker)
    data = stock.history(period="1d")

    if data.empty:
        raise ValueError(f"No data returned for ticker {ticker}")

    # Use the last close price
    spot_price = data["Close"].iloc[-1]
    return float(spot_price)


In [7]:
# Example usage
ticker = "AAPL"
S0 = get_spot_price(ticker)
print(f"Spot price for {ticker}: {S0:.2f}")

Spot price for AAPL: 273.76


Typical full pipeline:
S0 = get_spot_price("AAPL")
K = 180
T = 0.5      # years
r = 0.03     # 3%
sigma = 0.25 # 25%

→ feed into Black-Scholes / Monte Carlo / Binomial

But σ is beign assumed constant, which is not true in real life.
Here is the clean English version, suitable for documentation or a report:

* Historical volatility (σ): Calculated from the historical returns of the underlying asset (the selected ticker).

* Implied volatility (σ): Extracted from real market option prices using an option pricing model.

* Risk-free rate (r): Obtained from an interest rate source, such as government bond yields or a yield curve.
  Obtained from an interest rate source, such as government bond yields or a yield curve.

## Purpose get_hist_vol_annualized

This function computes **annualized historical volatility (σ)** of a stock using daily price data.

In finance:

> **Historical volatility** = how much the asset’s price has fluctuated in the past

> **Annualized** = scaled to a 1-year horizon so it can be used in option pricing models


### Choosing which price to use

Adj Close (Adjusted Close) is the adjusted closing price that takes into account dividends, splits, and other corporate actions, so that historical returns are comparable over time.

```python
if use_adj_close and "Adj Close" in df.columns:
    price_col = "Adj Close"
else:
    price_col = "Close"

prices = df[price_col].dropna()
```

Why this matters

* Adjusted Close:

  * Accounts for stock splits and dividends
  * Best for return calculations

* Close:

  * Raw market close price

The function:

* Uses Adjusted Close by default
* Falls back to Close if needed


## Computing Daily Log Returns

Given a price series $ P_t $, the **daily log return** is computed as:

$$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$

Here's how you can compute daily log returns in Python using NumPy and Pandas:

```python
log_returns = np.log(prices / prices.shift(1)).dropna()
```
**Steps:**

1. ( P_{t-1} ) is obtained with `prices.shift(1)`
2. Compute the price ratio ( P_t / P_{t-1} )
3. Apply the natural logarithm
4. Drop the first `NaN` value

**Why log returns?**

* Additive over time:

  $$
  \ln\left(\frac{P_T}{P_0}\right) = \sum_{t=1}^{T} r_t
  $$
* Standard in finance (volatility, Black–Scholes)
* For small changes:
  $$
  \ln(1+x) \approx x
  $$


### Computing volatility

```python
sigma = log_returns.std(ddof=1) * np.sqrt(trading_days)
```

This does two things:

#### 1. Daily volatility

```python
log_returns.std(ddof=1)
```

* Computes sample standard deviation of daily returns
* `ddof=1` → unbiased estimator (Delta Degrees of Freedom)

#### 2. Annualization

```python
* np.sqrt(trading_days)
```

Uses the volatility scaling rule:

$$
\sigma_{\text{annual}} = \sigma_{\text{daily}} \sqrt{252}
$$

### Returning the result

```python
return float(sigma.iloc[0])
```

* `sigma` is technically a pandas object
* `.iloc[0]` extracts the scalar value
* `float(...)` returns a clean Python float

This avoids future pandas errors.

> This function downloads daily stock prices, computes daily log returns, measures their variability, annualizes it, and returns historical volatility suitable for option pricing models.


In [ ]:
import numpy as np

def get_hist_vol_annualized(ticker, period="1y", use_adj_close = True, trading_days=252):
    """
    Annualized historical volatility (sigma) from daily log returns.
    period examples: "6mo", "1y", "2y", "5y"
    """
    df = yf.download(ticker, period=period, interval="1d", auto_adjust=False)
    if df.empty:
        raise ValueError(f"No price data for ticker {ticker}")

    if use_adj_close and "Adj Close" in df.columns:
        price_col = "Adj Close"
    else:
        price_col = "Close"

    prices = df[price_col].dropna()

    # Daily log returns
    log_returns = np.log(prices/prices.shift(1)).dropna()

    # Annualized sigma
    sigma = log_returns.std(ddof=1) * np.sqrt(trading_days)
    # return float(sigma)
    return float(sigma.iloc[0])

In [13]:
ticker = "AAPL"
sigma_hist = get_hist_vol_annualized(ticker, period="1y")
print("Historical sigma:", sigma_hist)

[*********************100%***********************]  1 of 1 completed

Historical sigma: 0.3228195400351139


## Purpose rolling_hist_vol
Computes the **rolling (moving) historical volatility** of a financial asset based on daily log returns.

- Measure how volatile an asset has been **recently**
- Track how volatility **evolves over time**

### Rolling Annualized Volatility

Given daily log returns \( r_t \), the **rolling annualized volatility** over a window of \( N \) days is:

$$
\sigma_t
= \sqrt{D} \;\cdot\;
\operatorname{Std}\left(r_{t-N+1}, \dots, r_t\right)
$$

where:
- $ N = \text{window} $: The size of the rolling window used for calculating standard deviation.
- $ D = \text{trading\_days} $ (typically 252): The number of trading days in a year.
- `Std(·)` is the **sample standard deviation**


In code:
```python
rolling_sigma = (
    log_returns
    .rolling(window)
    .std(ddof=1)
    * np.sqrt(trading_days)
)

rolling_sigma.name = f"{window}D_rolling_sigma"
```

**Steps:**

1. **`rolling(window)`**: Select the last $ N $ daily returns.
2. **`std(ddof=1)`**: Compute the sample standard deviation using:
   
   $$
   \sqrt{\frac{1}{N-1}\sum (r_i - \bar{r})^2}
   $$

3. Multiply by $ \sqrt{D} $ to **annualize** volatility.
4. Assign a descriptive name to the resulting series.

### Interpretation:

- $ \sigma_t $ is the **annualized volatility at time $ t $**.
- It is based only on the **previous $ N $ trading days**.
- For example, a value of $ 0.25 \Rightarrow 25\% $ annual volatility.




In [23]:
def rolling_hist_vol(ticker, period="1y", use_adj_close = True, trading_days=252, window=30):
    df=yf.download(ticker, period=period, interval="1d", auto_adjust=False, progress=False)
    if df.empty:
        raise ValueError(f"No price data for ticker {ticker}")
    
    if use_adj_close and "Adj Close" in df.columns:
        price_col = "Adj Close"
    else:
        price_col = "Close"
    prices = df[price_col].dropna()

    log_returns = np.log(prices/prices.shift(1)).dropna()
    rolling_sigma = log_returns.rolling(window).std(ddof=1) * np.sqrt(trading_days)
    rolling_sigma.name = f"{window}D_rolling_signma"

    return rolling_sigma    
    # return rolling_sigma.dropna().iloc[-1].item()


In [27]:
# Compute 30-day rolling historical volatility for Apple stock
sigma_30d_series = rolling_hist_vol(
    ticker="AAPL",
    period="1y",
    window=30,
    trading_days=252
)

# Extract the most recent volatility value
latest_sigma_30d = sigma_30d_series.dropna().iloc[-1].item()
print("Sigma Series: \n", sigma_30d_series)


print("Latest 30D rolling volatility:", latest_sigma_30d)

Sigma Series: 
 Ticker          AAPL
Date                
2024-12-31       NaN
2025-01-02       NaN
2025-01-03       NaN
2025-01-06       NaN
2025-01-07       NaN
...              ...
2025-12-22  0.154581
2025-12-23  0.154727
2025-12-24  0.142276
2025-12-26  0.141082
2025-12-29  0.141015

[249 rows x 1 columns]
Latest 30D rolling volatility: 0.14101504427325967


And why do NaN values ​​appear at the beginning of the series?

Because with a 30-day window, the first 29 values ​​don't have enough data to calculate the standard deviation → they appear as NaN. That's normal.

```
Sigma Series: 
 Ticker          AAPL
Date                
2024-12-31       NaN
2025-01-02       NaN
2025-01-03       NaN
2025-01-06       NaN
2025-01-07       NaN
...              ...
2025-12-22  0.154581
2025-12-23  0.154727
2025-12-24  0.142276
2025-12-26  0.141082
2025-12-29  0.141015

[249 rows x 1 columns]
Latest 30D rolling volatility: 0.14101504427325967
```